# MongoDB marketing database

Explore the `marketing` database created for this project. MongoDB collections are the equivalent of tables: `campaigns`, `ad_performance`, and `store_visits`. This notebook is read-only; it does not modify database contents.

Start the local MongoDB server before connecting. Install notebook dependencies if needed with `%pip install pymongo pandas`.

In [ ]:
import os
from decimal import Decimal

import pandas as pd
from IPython.display import display
from pymongo import MongoClient
from bson.decimal128 import Decimal128

MONGODB_URI = os.getenv("MONGODB_URI", "mongodb://127.0.0.1:27017")
DATABASE_NAME = "marketing"

In [ ]:
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
client.admin.command("ping")
db = client[DATABASE_NAME]
print(f"Connected to {DATABASE_NAME!r} at {MONGODB_URI}")
print("Collections:", sorted(db.list_collection_names()))

## Collection counts and sample documents

In [ ]:
collection_names = ["campaigns", "ad_performance", "store_visits"]
counts = pd.DataFrame([
    {"collection": name, "documents": db[name].count_documents({})}
    for name in collection_names
])
display(counts)

for name in collection_names:
    print(f"\n{name} — first 3 documents")
    sample = list(db[name].find({}, {"_id": 0}).limit(3))
    display(pd.json_normalize(sample) if sample else pd.DataFrame())

## Advertising totals by source

Money and fractional values use BSON `Decimal128`; convert them to Python `Decimal` to retain precision during totals. Currency codes are grouped separately because the Meta source can contain currencies other than USD.

In [ ]:
def to_decimal(value):
    if value is None:
        return None
    if isinstance(value, Decimal128):
        return value.to_decimal()
    return Decimal(str(value))

projection = {
    "_id": 0,
    "source": 1,
    "currency": 1,
    "metrics.spend_usd": 1,
    "metrics.conversions": 1,
    "metrics.conversion_value_usd": 1,
}
ad_df = pd.json_normalize(list(db.ad_performance.find({}, projection)))
metric_columns = [
    "metrics.spend_usd",
    "metrics.conversions",
    "metrics.conversion_value_usd",
]
for column in metric_columns:
    ad_df[column] = ad_df[column].map(to_decimal)

ad_summary = ad_df.groupby(["source", "currency"], dropna=False).agg(
    records=("source", "size"),
    spend=("metrics.spend_usd", lambda values: sum((v for v in values if v is not None), Decimal("0"))),
    conversions=("metrics.conversions", lambda values: sum((v for v in values if v is not None), Decimal("0"))),
    conversion_value=("metrics.conversion_value_usd", lambda values: sum((v for v in values if v is not None), Decimal("0"))),
).reset_index()
display(ad_summary)

## Daily store visits

Store visits remain separate from campaign results because their source data has no campaign identifier.

In [ ]:
visit_rows = list(db.store_visits.find(
    {},
    {"_id": 0, "date": 1, "attribution_window_days": 1, "attributed_visits": 1},
))
visits_df = pd.DataFrame(visit_rows)
visits_df["date"] = pd.to_datetime(visits_df["date"], utc=True)
daily_visits = visits_df.groupby(
    ["date", "attribution_window_days"], as_index=False
)["attributed_visits"].sum().sort_values(
    ["date", "attribution_window_days"]
)
display(daily_visits.head(20))

## Close the database connection

Run this cell after exploration. It closes the client and does not change the database.

In [ ]:
client.close()
print("MongoDB client closed.")